In [17]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [18]:
df = pd.read_csv("../data/spotify_2015_2025_85k.csv")
df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,0,-14.02,0,0.223,157.74,1000,France,1,Island Records
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,10,-17.84,0,0.304,155.85,2000,United States,0,Independent


In [35]:
df.info()
df.isnull().sum()
df.columns

<class 'pandas.DataFrame'>
Index: 84979 entries, 0 to 84999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          84979 non-null  str    
 1   track_name        84979 non-null  str    
 2   artist_name       84979 non-null  str    
 3   album_name        84933 non-null  str    
 4   release_date      84979 non-null  str    
 5   genre             84979 non-null  str    
 6   duration_ms       84979 non-null  int64  
 7   popularity        84979 non-null  int64  
 8   danceability      84979 non-null  float64
 9   energy            84979 non-null  float64
 10  key               84979 non-null  int64  
 11  loudness          84979 non-null  float64
 12  mode              84979 non-null  int64  
 13  instrumentalness  84979 non-null  float64
 14  tempo             84979 non-null  float64
 15  stream_count      84979 non-null  int64  
 16  country           84979 non-null  str    
 17  explicit 

Index(['track_id', 'track_name', 'artist_name', 'album_name', 'release_date',
       'genre', 'duration_ms', 'popularity', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'instrumentalness', 'tempo', 'stream_count',
       'country', 'explicit', 'label'],
      dtype='str')

In [20]:
# since genre is in the form of different unique genres
# use OneHotEncoding to make each genre into booleans 
genre_encoded = pd.get_dummies(df["genre"])

In [21]:
features = df[[
    'danceability', 'energy', 'loudness', 'instrumentalness', 'tempo', 
    'key', 'mode', 'popularity'
    # ,'duration_ms'
]]

X = pd.concat([features, genre_encoded], axis=1)

In [22]:
# drop rows with missing values (track_name)
df = df.dropna(subset = ['track_name'])

In [23]:
# make data into scaled bc knn depends on the distance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [24]:
# use euclidean distance

knn = NearestNeighbors(
    n_neighbors=10,
    metric="euclidean"
)

knn.fit(X_scaled)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [25]:
# lookup track by track_name
def get_song_index(song_name):
    return df[df["track_name"] == song_name].index[0]

In [26]:
# get la neighbors for that song
def get_neighbors(song_name):
    index = get_song_index(song_name)
    song_vector = X_scaled[index].reshape(1,-1)
    distances, indices = knn.kneighbors(song_vector)

    return distances[0], indices[0]

In [ ]:
def genre_bonus(index, song_A, song_B):
    genre_A = df[df["track_name"] == song_A]["genre"].values[0]
    genre_B = df[df["track_name"] == song_B]["genre"].values[0]

    genre_C = df.iloc[index]["genre"]

    bonus = 0

    if genre_C == genre_A:
        bonus += 0.1
    if genre_C == genre_B:
        bonus += 0.1

    return bonus

In [28]:
def bridge_recommendation(song_A, song_B, top_n=5): 
    dist_A, ind_A = get_neighbors(song_A) # the distance and the independent variables chosen
    dist_B, ind_B = get_neighbors(song_B)

    scores = {}

    # convert those distances into similarity
    for d, i in zip(dist_A, ind_A):
        # scores[i] = scores.get(i, 0) + (1 / (1 + d)) # without the added weight of genre(OneHotEncoding)
        scores[i] = scores.get(i, 0) + (1 / (1 + d)) + genre_bonus(i, song_A, song_B)

    for d, i in zip(dist_B, ind_B):
        # scores[i] = scores.get(i, 0) + (1 / (1 + d))
        scores[i] = scores.get(i, 0) + (1 / (1 + d)) + genre_bonus(i, song_A, song_B)

    # sort by combined score
    sorted_songs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for i, score in sorted_songs:
        name = df.iloc[i]["track_name"]

        if name not in [song_A, song_B]:
            results.append((name, score))

    return results[:top_n]

In [29]:
def normalize_scores(results):
    if not results:
        return []
    
    max_score = max(score for _, score in results)

    output = []
    for name, score in results:
        confidence = score / max_score
        output.append((name, score, confidence))

    return output

In [30]:
results = bridge_recommendation("Husband at", "Right")
results


Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal
Country
Metal


[('Food behavior concern', np.float64(0.614988257341276)),
 ('Rate walk', np.float64(0.5756583266211057)),
 ('War', np.float64(0.5662405969315244)),
 ('Camera cold debate', np.float64(0.5561671287154615)),
 ('Plan table meet', np.float64(0.5529112046422671))]

In [31]:
normalized_results = normalize_scores(results)
normalized_results

[('Food behavior concern', np.float64(0.614988257341276), np.float64(1.0)),
 ('Rate walk', np.float64(0.5756583266211057), np.float64(0.9360476720479153)),
 ('War', np.float64(0.5662405969315244), np.float64(0.9207339980433153)),
 ('Camera cold debate',
  np.float64(0.5561671287154615),
  np.float64(0.9043540621732995)),
 ('Plan table meet',
  np.float64(0.5529112046422671),
  np.float64(0.899059775600625))]

In [32]:
# score is the raw value and confidence is the normalized value (0-1)
for name, score, confidence in normalized_results:
    print(f"{name} → Score: {score:.4f} | Confidence: {confidence:.2%}")

Food behavior concern → Score: 0.6150 | Confidence: 100.00%
Rate walk → Score: 0.5757 | Confidence: 93.60%
War → Score: 0.5662 | Confidence: 92.07%
Camera cold debate → Score: 0.5562 | Confidence: 90.44%
Plan table meet → Score: 0.5529 | Confidence: 89.91%
